In [1]:
import torch
import os

# =====================================================================
# 🛠️ KONFIGURATION
# =====================================================================
# Trage hier die Pfade zu deinen letzten/besten 3 bis 5 Checkpoints 
# aus der ganz flachen Decay-Phase ein.
checkpoint_paths = [
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_best.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_59136.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_57904.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_56672.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_55440.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_54208.pth"
]

# Der Name der neuen, gemittelten Datei
output_path = "checkpoints/checkpoint_v3_1_best_swa.pth"

# =====================================================================
# 🚀 SWA LOGIK
# =====================================================================
def create_swa_model(filepaths, out_path):
    print(f"🔄 Starte SWA: Mittle {len(filepaths)} Checkpoints...")
    
    swa_state_dict = None
    num_checkpoints = len(filepaths)
    
    for path in filepaths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"❌ Checkpoint nicht gefunden: {path}")
            
        print(f"📥 Lade: {path}")
        # Wir laden direkt auf die CPU ('cpu'), das verhindert VRAM-Überläufe
        ckpt = torch.load(path, map_location='cpu')
        
        # Extrahiere das model_state_dict (ignoriere Optimizer, Scheduler etc.)
        model_state = ckpt.get('model_state_dict', ckpt)
        
        if swa_state_dict is None:
            # Beim ersten Checkpoint: Dictionary tief kopieren (.clone())
            swa_state_dict = {k: v.clone() for k, v in model_state.items()}
        else:
            # Bei allen weiteren: Gewichte einfach aufaddieren
            for k in swa_state_dict.keys():
                if k in model_state:
                    swa_state_dict[k] += model_state[k]
                else:
                    print(f"⚠️ Warnung: Key {k} fehlt in {path}!")
    
    # 🧮 Durchschnitt berechnen (Teilen durch N)
    print("🧮 Berechne den mathematischen Durchschnitt...")
    for k in swa_state_dict.keys():
        if swa_state_dict[k].is_floating_point():
            # Float-Tensoren (Gewichte, Biases) normal teilen
            swa_state_dict[k].div_(num_checkpoints)
        else:
            # Integer-Tensoren (z.B. num_batches_tracked in BatchNorms) 
            # dürfen keine Kommazahlen werden -> Floor Division
            swa_state_dict[k] = torch.div(swa_state_dict[k], num_checkpoints, rounding_mode='floor')
            
    # 💾 Speichern
    print(f"💾 Speichere SWA-Modell unter: {out_path}")
    # Wir speichern NUR die Gewichte, da dieses Modell fertig trainiert ist
    torch.save({'model_state_dict': swa_state_dict}, out_path)
    print("✅ SWA erfolgreich abgeschlossen! Das Modell ist bereit für den Hailo-Export.")

# Ausführen
if __name__ == "__main__":
    create_swa_model(checkpoint_paths, output_path)



🔄 Starte SWA: Mittle 6 Checkpoints...
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_best.pth


/tmp/ipykernel_1701259/3708634092.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location='cpu')


📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_59136.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_57904.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_56672.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_55440.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_54208.pth
🧮 Berechne den mathematischen Durchschnitt...
💾 Speichere SWA-Modell unter: checkpoints/checkpoint_v3_1_best_swa.pth
✅ SWA erfolgreich abgeschlossen! Das Modell ist bereit für den Hailo-Export.
